# Ce notebook n'est pas vraiment nécessaire au rapport c'est simplement pour garder la petite analyse exploratrice du jeu de donnée que nous faisions dans fetch_clean avant lorsque nous allions chercher les données pour GLIMS, nous voulons la faire ici afin d'alléger le notebook data_fetch_clean et aussi parce que ce sont ces graphiques qui nous ont en grande partie découragée de faire une tache de prédiciton (ce que nous voulions initialement faire avec le projet) et qui sont plus forcément aussi pertinent maintenant que nous a transitionner vers une tâche de segmentation

### Remarque: pour l'instant j'ai juste copier coller l'ancien pipeline mais il faudrait faire quelque chose de plus narratif où on montre que c'était pas vraiment envisageable

# GLIMS
## Fetch les données brutes
### Une fois téléchargées, elles seront dans /projet_glacier/data/raw/...

In [ ]:
from glacier.data import fetch_data, unzip_to
paths = fetch_data("20260114")

#### Dézippe les fichiers téléchargers


In [ ]:
paths = fetch_data("20260114")
extracted_root = paths[0].parent / "extracted"
extracted_dirs = unzip_to(paths, extracted_root)

## Exploration du jeu de donnée brute

In [ ]:
from pathlib import Path
from glacier.data.data_fetching import repo_root
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
root = repo_root()
base = root / "data" / "raw" / "glims_v1" / "extracted"

north = next(p for p in base.rglob("glims_polygons.shp") if "_north_" in str(p).lower())
south = next(p for p in base.rglob("glims_polygons.shp") if "_south_" in str(p).lower())

gdf_raw_north = gpd.read_file(north)
gdf_raw_south = gpd.read_file(south)
gdf_raw = pd.concat([gdf_raw_north, gdf_raw_south])
len(gdf_raw), gdf_raw.crs #.crs est pour vérifier le système de coordonnéees 

In [ ]:
gdf_raw.columns
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)
gdf_raw.head(5)

In [ ]:
gdf_raw.info()

In [ ]:
gdf_raw[["min_elev","mean_elev","max_elev"]].describe()

In [ ]:
df_raw = pd.DataFrame(gdf_raw.drop(columns="geometry")) # moins lourd pour certains graphes

df = df_raw.copy()
df["src_date_dt"] = pd.to_datetime(df["src_date"], errors="coerce")

In [ ]:
years = df["src_date_dt"].dt.year

plt.figure(figsize=(10,4))
sns.histplot(years.dropna(), bins=120)
plt.title("Distribution des obserations selon l'année")
plt.xlabel("Année")
plt.ylabel("Nombre d'observations")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10,4))
order = df["line_type"].value_counts().head(10).index  # top 10 si jamais il y en a beaucoup
sns.countplot(data=df, x="line_type", order=order)
plt.title("Répartition des line_type)")
plt.xlabel("line_type (top 12)")
plt.ylabel("Nombre d'observations")
plt.xticks(rotation=35, ha="right")
plt.tight_layout()
plt.show()

In [ ]:
if "area" in df.columns:
    plt.figure(figsize=(10,4))
    sns.histplot(df["area"], bins=120)
    plt.title("Distribution de area")
    plt.xlabel("area")
    plt.ylabel("Nombre d'observations")
    plt.tight_layout()
    plt.show()

    # version log (plus lisible si très skew)
    positive = df["area"].where(df["area"] > 0)
    plt.figure(figsize=(10,4))
    sns.histplot(np.log10(positive.dropna()), bins=80)
    plt.title("Distribution de log10(area)")
    plt.xlabel("log10(area)")
    plt.ylabel("Nombre d'observations")
    plt.tight_layout()
    plt.show()

In [ ]:
tmp = df.dropna(subset=["glac_id"]).copy()
n_outlines = tmp.groupby("glac_id").size()

tmp2 = tmp.dropna(subset=["src_date_dt"])
n_dates = tmp2.groupby("glac_id")["src_date_dt"].nunique()

cap = 10

plt.figure(figsize=(10,4))
sns.histplot(n_outlines.clip(upper=cap), bins=cap, discrete=True)
plt.title("Nombre d'outlines par glacier")
plt.xlabel("n_outlines")
plt.ylabel("Nombre de glaciers")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10,4))
sns.histplot(n_dates.clip(upper=cap), bins=cap, discrete=True)
plt.title("Nombre de dates uniques par glacier")
plt.xlabel("n_unique_dates")
plt.ylabel("Nombre de glaciers")
plt.tight_layout()
plt.show()

# Nettoyage du jeu de données
## Une fois téléchargées, elles seront dans /projet_glacier/data/processed/...
Pour voir précisément comment les nettoyage est fait, il est possible de consulter le fichier
src/glacier/data/data_cleaning.py.
Pour faire court, le nettoyage du jeu de données consiste à :
- Conserver uniquement les contours de glaciers pertinents (line_type = "glac_bound").
- Nettoyer les géométries (retirer les géométries vides/nulles, définir le CRS et corriger les géométries invalides) pour fiabiliser les opérations spatiales. (il y en a quasiment aucunes mais juste au cas où)
- Recoder les valeurs sentinelles (p. ex. -9999) des variables d’élévation en valeurs manquantes (NaN) sans supprimer d’observations.
- Supprimer les valeurs non physiques (aire ≤ 0) et les doublons exacts.

Ensuite nous filtrons de façons à garder que ce qui semble exploitable pour pouvoir faire de la prédiction

- Nous filtrons les glaciers disposant d’au moins $k\geq 2$ dates d’observation distinctes afin de disposer d’une information temporelle suffisante
- Convertir src_date en datetime et gérer les valeurs aberrantes; le filtrage temporel (p. ex. ≥1900 ou ≥2015) est appliqué dans une vue dédiée à l’analyse temporelle.

In [ ]:
from glacier.data import fetch_data, clean_glims_outlines, make_temporal_view

In [ ]:
gdf_north = clean_glims_outlines(gdf_raw_north)
gdf_south = clean_glims_outlines(gdf_raw_south)
gdf = gpd.GeoDataFrame(pd.concat([gdf_north, gdf_south], ignore_index=True), crs=gdf_north.crs)

In [ ]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", None)

gdf.head()

In [ ]:
print("raw north:", len(gdf_raw_north), "-> clean:", len(gdf_north)) 
print("raw south:", len(gdf_raw_south), "-> clean:", len(gdf_south))
print("total clean:", len(gdf))
gdf.info()

In [ ]:
gdf[["min_elev","mean_elev","max_elev"]].describe()

In [ ]:
temporal = make_temporal_view(gdf, min_date="1900-01-01")

In [ ]:
root = repo_root()
proc = root / "data" / "processed"
proc.mkdir(parents=True, exist_ok=True)

(gdf).to_parquet(proc / "glims_base.parquet")
temporal.to_parquet(proc / "glims_temporal_1900.parquet")

### pour recuperer les jeu de données dans les prochains notebook sans tout rerun on fera 
# gdf = gpd.read_parquet(proc / "glims_base.parquet")
# pred = gpd.read_parquet(proc / "glims_pred_1900.parquet")
# pred_s2 = gpd.read_parquet(proc / "glims_pred_s2.parquet")

In [ ]:
# Commençons alors par filtrer le datafram glims pour avoir que les glaciers des régions mentionnées
from glacier.data import filter_regions

temporal_reg = filter_regions(temporal)  # alps + caucasus + andes
temporal_reg.to_parquet(proc / "glims_temporal_regions.parquet")

In [ ]:
root = repo_root()
temporal_reg = gpd.read_parquet(root / "data" / "processed"/ "glims_temporal_regions.parquet")
temporal_reg

In [ ]:
temporal_sorted = temporal_reg.sort_values("src_date_dt")
glaciers_ref = temporal_sorted.drop_duplicates("glac_id", keep="last").copy()
print("Nb glaciers uniques:", glaciers_ref["glac_id"].nunique())

In [ ]:
glaciers_ref = glaciers_ref[glaciers_ref["area"] >= 0.05].copy()
glaciers_ref

In [ ]:

SEED = 0
YEARS = [2019, 2020, 2021, 2022, 2023, 2024]

glaciers_ref = glaciers_ref.copy()
glaciers_ref["year_img"] = None

rng = np.random.default_rng(SEED)
for reg in glaciers_ref["region"].unique():
    idx = glaciers_ref.index[glaciers_ref["region"] == reg].to_numpy()
    y_assigned = np.resize(np.array(YEARS), idx.shape[0])
    rng.shuffle(y_assigned)
    glaciers_ref.loc[idx, "year_img"] = y_assigned

glaciers_ref["year_img"] = glaciers_ref["year_img"].astype(int)

In [ ]:
MAX_GAP = 3

gl = temporal_reg[["glac_id","region","src_date_dt","geometry","area"]].copy()
gl["glac_id"] = gl["glac_id"].astype(str).str.strip()
gl["src_year"] = gl["src_date_dt"].dt.year

s = glaciers_ref[["glac_id","region","year_img"]].copy()
s["glac_id"] = s["glac_id"].astype(str).str.strip()

m = s.merge(gl, on=["glac_id","region"], how="left")
m["gap"] = (m["src_year"] - m["year_img"]).abs()

closest = (m.sort_values(["glac_id","gap"])
            .drop_duplicates("glac_id", keep="first")
            .copy())

pool = closest[closest["gap"] <= MAX_GAP].copy()
print(pool["gap"].describe())